# Lumina3D Backend Server (Colab / VS Code Notebook)

This notebook supports two modes:
- **Colab remote runtime** (recommended for GPU)
- **Local kernel** (debug only, usually no GPU).

Run cells top to bottom.

In [ ]:
# 1) Detect runtime mode
import os
import sys
from pathlib import Path

IS_COLAB = 'COLAB_RELEASE_TAG' in os.environ or Path('/content').exists()
print('IS_COLAB =', IS_COLAB)
print('Python executable =', sys.executable)
if not IS_COLAB:
    print('Local kernel detected. This mode is fine for API wiring checks, but full model inference may fail without GPU and native deps.')

try:
    import torch
    print('torch.__version__ =', torch.__version__)
    print('torch.cuda.is_available() =', torch.cuda.is_available())
except Exception as exc:
    print('Torch import failed in notebook kernel:', exc)


In [ ]:
# 2) Install base dependencies (lean + deterministic)
%pip install fastapi "uvicorn[standard]" python-multipart pyngrok requests psutil opencv-python Pillow trimesh

import importlib.util
import subprocess
import sys

if importlib.util.find_spec('torch') is None:
    print('torch not found, installing torch...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch'], check=False)
else:
    print('torch already present; skipping torch install')


In [ ]:
# 3) Resolve project directories
import os
from pathlib import Path
import subprocess

if IS_COLAB:
    LUMINA_REPO = os.getenv('LUMINA_REPO_URL', 'https://github.com/AravindS2006/Lumina3D.git')
    LUMINA_BRANCH = os.getenv('LUMINA_REPO_BRANCH', 'main')
    LUMINA_ROOT = Path('/content/Lumina3D')
    if LUMINA_ROOT.exists():
        subprocess.run(['git', '-C', str(LUMINA_ROOT), 'fetch', '--all'], check=False)
        subprocess.run(['git', '-C', str(LUMINA_ROOT), 'checkout', LUMINA_BRANCH], check=False)
        subprocess.run(['git', '-C', str(LUMINA_ROOT), 'pull', 'origin', LUMINA_BRANCH], check=False)
    else:
        subprocess.run(['git', 'clone', '-b', LUMINA_BRANCH, LUMINA_REPO, str(LUMINA_ROOT)], check=True)
else:
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path.cwd() / 'Lumina3D',
    ]
    LUMINA_ROOT = None
    for c in candidates:
        if (c / 'backend' / 'app' / 'main.py').exists():
            LUMINA_ROOT = c
            break
    if LUMINA_ROOT is None:
        raise FileNotFoundError('Could not find Lumina3D root. Open notebook from inside the repo.')

BACKEND_DIR = LUMINA_ROOT / 'backend'
print('LUMINA_ROOT =', LUMINA_ROOT)
print('BACKEND_DIR =', BACKEND_DIR)
main_py = BACKEND_DIR / 'app' / 'main.py'
print('Backend entry exists =', main_py.exists())
main_code = main_py.read_text(encoding='utf-8', errors='ignore')
print('Contains /debug/runtime route =', '/debug/runtime' in main_code)
print('Contains runtime_probe in /healthz =', 'runtime_probe' in main_code)
if '/debug/runtime' not in main_code or 'runtime_probe' not in main_code:
    raise RuntimeError('Stale backend source detected. Update runtime repo to latest code (push local changes to remote, then rerun this cell).')

In [ ]:
# 4) Optional: clone/update Hunyuan repos (Colab mode only)
from pathlib import Path
import subprocess

if IS_COLAB:
    HY21_REPO = 'https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1.git'
    HY20_REPO = 'https://github.com/Tencent/Hunyuan3D-2.git'
    HY21_ROOT = Path('/content/Hunyuan3D-2.1')
    HY20_ROOT = Path('/content/Hunyuan3D-2')

    if HY21_ROOT.exists():
        subprocess.run(['git', '-C', str(HY21_ROOT), 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', HY21_REPO, str(HY21_ROOT)], check=True)

    if HY20_ROOT.exists():
        subprocess.run(['git', '-C', str(HY20_ROOT), 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', HY20_REPO, str(HY20_ROOT)], check=True)

    print('HY21_ROOT =', HY21_ROOT)
    print('HY20_ROOT =', HY20_ROOT)
else:
    HY21_ROOT = Path(os.getenv('LUMINA_HUNYUAN21_ROOT', '')) if os.getenv('LUMINA_HUNYUAN21_ROOT') else None
    HY20_ROOT = Path(os.getenv('LUMINA_HUNYUAN2_ROOT', '')) if os.getenv('LUMINA_HUNYUAN2_ROOT') else None
    print('Using existing env paths for Hunyuan repos in local mode.')

In [ ]:
# 5) Optional: install Hunyuan requirements (heavy, can take a long time)
import os
import subprocess

INSTALL_HEAVY_REQS = os.getenv('LUMINA_INSTALL_HEAVY_REQS', '0') == '1'
if IS_COLAB or INSTALL_HEAVY_REQS:
    reqs = [
        '/content/Hunyuan3D-2/requirements.txt',
        '/content/Hunyuan3D-2.1/requirements.txt',
    ]
    for req in reqs:
        cmd = ['python', '-m', 'pip', 'install', '-r', req]
        print('Running:', ' '.join(cmd))
        subprocess.run(cmd, check=False)
    print('Requirement pass complete.')
else:
    print('Skipping heavy Hunyuan requirements. Set LUMINA_INSTALL_HEAVY_REQS=1 to force this step.')


In [ ]:
# 6) Configure environment
import os
from getpass import getpass

if IS_COLAB:
    os.environ['LUMINA_HUNYUAN21_ROOT'] = '/content/Hunyuan3D-2.1'
    os.environ['LUMINA_HUNYUAN2_ROOT'] = '/content/Hunyuan3D-2'

os.environ['CORS_ALLOW_ORIGINS'] = 'http://localhost:5173,http://127.0.0.1:5173,http://192.168.0.105:5173'
os.environ['ENABLE_NGROK'] = '0'

if not os.environ.get('NGROK_AUTHTOKEN'):
    token = getpass('NGROK_AUTHTOKEN (leave blank to skip tunnel now): ').strip()
    if token:
        os.environ['NGROK_AUTHTOKEN'] = token

print('Env configured.')

In [ ]:
# 7) Start backend (cross-platform, force clean port 8000)
import json
import os
import signal
import subprocess
import sys
import tempfile
import time
from pathlib import Path

import psutil

state_path = Path(tempfile.gettempdir()) / 'lumina3d_server_state.json'
log_path = Path(tempfile.gettempdir()) / 'lumina3d.log'

# 7a) stop previously tracked process
if state_path.exists():
    try:
        state = json.loads(state_path.read_text())
        old_pid = int(state.get('pid', 0))
        if old_pid > 0 and psutil.pid_exists(old_pid):
            psutil.Process(old_pid).terminate()
            time.sleep(1)
    except Exception:
        pass

# 7b) kill any process still bound to port 8000
pids_on_8000 = set()
for conn in psutil.net_connections(kind='inet'):
    try:
        if conn.laddr and conn.laddr.port == 8000 and conn.pid:
            pids_on_8000.add(conn.pid)
    except Exception:
        pass

for pid in pids_on_8000:
    try:
        psutil.Process(pid).terminate()
    except Exception:
        pass
time.sleep(1)
for pid in list(pids_on_8000):
    try:
        if psutil.pid_exists(pid):
            psutil.Process(pid).kill()
    except Exception:
        pass

cmd = [
    sys.executable,
    '-m',
    'uvicorn',
    '--app-dir',
    str(BACKEND_DIR),
    'app.main:app',
    '--host',
    '127.0.0.1',
    '--port',
    '8000',
    '--log-level',
    'info',
]

log_handle = open(log_path, 'w', encoding='utf-8')
proc = subprocess.Popen(cmd, stdout=log_handle, stderr=subprocess.STDOUT, cwd=str(BACKEND_DIR))
state_path.write_text(json.dumps({'pid': proc.pid, 'log_path': str(log_path)}))

for _ in range(15):
    if proc.poll() is not None:
        break
    time.sleep(1)

print('Killed stale PIDs on 8000:', sorted(pids_on_8000))
print('Backend PID:', proc.pid)
print('Backend running:', proc.poll() is None)
print('Log file:', log_path)
if log_path.exists():
    lines = log_path.read_text(encoding='utf-8', errors='ignore').splitlines()
    print('\n'.join(lines[-120:]))

if proc.poll() is not None:
    raise RuntimeError('Backend process exited early. Check logs above.')

In [ ]:
# 8) Probe backend readiness
import requests
import time

base_local = 'http://127.0.0.1:8000'
last_error = None
for _ in range(20):
    try:
        r = requests.get(base_local + '/healthz', timeout=5)
        print('healthz:', r.status_code, r.text)
        payload = r.json() if r.headers.get('content-type', '').startswith('application/json') else {}
        if payload.get('runtime_probe') is not True:
            raise RuntimeError('Outdated backend process detected: /healthz missing runtime_probe=true')
        print('API cuda_available =', payload.get('cuda_available'))
        probe = requests.get(base_local + '/debug/runtime', timeout=10).json()
        print('runtime probe:', probe)
        try:
            import torch
            print('Notebook torch.cuda.is_available() =', torch.cuda.is_available())
        except Exception as exc:
            print('Notebook torch import failed during probe:', exc)
        last_error = None
        break
    except Exception as exc:
        last_error = exc
        time.sleep(1)

if last_error is not None:
    raise RuntimeError(f'Backend did not become ready: {last_error}')


In [ ]:
# 9) Optional: create ngrok tunnel
from pyngrok import ngrok
import requests
import os

token = os.environ.get('NGROK_AUTHTOKEN', '').strip()
if not token:
    print('NGROK_AUTHTOKEN not set. Skipping tunnel creation.')
else:
    ngrok.kill()
    ngrok.set_auth_token(token)
    tunnel = ngrok.connect(addr='127.0.0.1:8000', bind_tls=True)
    public_url = tunnel.public_url
    headers = {'ngrok-skip-browser-warning': '69420'}
    h = requests.get(public_url + '/healthz', headers=headers, timeout=15)
    g = requests.get(public_url + '/generate', headers=headers, timeout=15)
    print('ngrok:', public_url)
    print('healthz status:', h.status_code)
    print('generate status (expect 405 for GET):', g.status_code)
    print('Set frontend env -> VITE_API_BASE_URL=' + public_url)


## Notes

- If backend fails to start, check the printed log from cell 7.
- In local kernel mode, use this primarily for API wiring tests. Full Hunyuan inference is best on Colab GPU runtime.
- Cell 5 is intentionally optional/heavy. Set `LUMINA_INSTALL_HEAVY_REQS=1` to force that install pass.
- ngrok tunnel targets `127.0.0.1:8000` explicitly to avoid upstream routing mismatches.
